## Phase 1 : Setup et Imports

In [ ]:
# Imports nécessaires
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import (
    train_test_split,
    cross_val_score,
    StratifiedKFold,
    GridSearchCV,
)
from sklearn.preprocessing import StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import (
    RandomForestClassifier,
    VotingClassifier,
    StackingClassifier,
)
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
    roc_curve,
    auc,
)
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
import pickle
import warnings

warnings.filterwarnings("ignore")

# Download NLTK resources
nltk.download("stopwords", quiet=True)
nltk.download("punkt", quiet=True)
nltk.download("wordnet", quiet=True)

# Configuration
np.random.seed(42)
plt.style.use("seaborn-v0_8-darkgrid")
sns.set_palette("husl")

print("Imports complétés avec succès!")

## Phase 2 : Chargement et Exploration des Données

In [ ]:
# Charger les données Amazon Reviews
# NOTE: Télécharger depuis https://www.kaggle.com/datasets/bittlingmayer/amazonreviews
# Pour ce notebook, on utilise une version simplifiée


def load_amazon_data(filepath, max_samples=5000):
    """Charger les données Amazon Reviews au format .ft.txt"""
    reviews = []
    labels = []

    with open(filepath, "r", encoding="utf-8") as f:
        for i, line in enumerate(f):
            if i >= max_samples:
                break
            # Format: __label__1 or __label__2 + texte
            parts = line.strip().split(" ", 1)
            if len(parts) == 2:
                label = 1 if "__label__2" in parts[0] else 0  # Positif=1, Négatif=0
                text = parts[1]
                reviews.append(text)
                labels.append(label)

    return pd.DataFrame({"text": reviews, "label": labels})


# Alternative: Créer des données de démonstration
data = pd.DataFrame(
    {
        "text": [
            "Excellent product, highly recommend",
            "Very bad quality, waste of money",
            "Amazing! Will buy again",
            "Terrible experience with customer service",
            "Love it, perfect quality",
            "Horrible product, broke after one day",
            "Outstanding value for money",
            "Disappointing, not as described",
            "Fantastic quality, fast shipping",
            "Poor quality, broke immediately",
        ]
        * 500,  # Répéter pour avoir 5000 avis
        "label": [1, 0, 1, 0, 1, 0, 1, 0, 1, 0] * 500,
    }
)

print(f"Données chargées: {data.shape}")
print(f"\nDistribution classes:")
print(data["label"].value_counts())
print(f"\nPremiers avis:")
print(data.head())

## Phase 3 : Exploration Exploratoire (EDA)

In [ ]:
# Statistiques basiques
print("=== EXPLORATION DES DONNÉES ===")
print(f"\nTaille dataset: {len(data)}")
print(f"Nombre unique avis: {data['text'].nunique()}")

# Longueur des avis
data["text_length"] = data["text"].apply(len)
data["word_count"] = data["text"].apply(lambda x: len(x.split()))

print(f"\nLongueur moyenne: {data['text_length'].mean():.0f} caractères")
print(f"Longueur min/max: {data['text_length'].min()} / {data['text_length'].max()}")
print(f"Nombre de mots moyen: {data['word_count'].mean():.1f}")

# Distribution par classe
print(f"\nDistribution classes:")
print(data["label"].value_counts(normalize=True))

# Visualisations
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

# Classe distribution
data["label"].value_counts().plot(kind="bar", ax=axes[0, 0], color=["red", "green"])
axes[0, 0].set_title("Distribution des Sentiments")
axes[0, 0].set_xticklabels(["Négatif", "Positif"], rotation=0)

# Longueur distribution
axes[0, 1].hist(data["text_length"], bins=30, color="blue", alpha=0.7)
axes[0, 1].set_title("Distribution Longueur Avis")
axes[0, 1].set_xlabel("Nombre de caractères")

# Word count
axes[1, 0].hist(data["word_count"], bins=30, color="orange", alpha=0.7)
axes[1, 0].set_title("Distribution Nombre de Mots")
axes[1, 0].set_xlabel("Nombre de mots")

# Longueur par sentiment
data.boxplot(column="text_length", by="label", ax=axes[1, 1])
axes[1, 1].set_title("Longueur par Sentiment")
axes[1, 1].set_xticklabels(["Négatif", "Positif"])

plt.tight_layout()
plt.show()

print("\nEDA complétée!")

## Phase 4 : Prétraitement et Nettoyage

In [ ]:
import re
from html import unescape


def preprocess_text(text):
    """Nettoyer et préparer le texte pour la modélisation"""
    # Minuscules
    text = text.lower()

    # Suppression HTML entities
    text = unescape(text)

    # Suppression URLs
    text = re.sub(r"http\S+|www\S+|https\S+", "", text, flags=re.MULTILINE)

    # Suppression mentions
    text = re.sub(r"@\w+|#\w+", "", text)

    # Suppression caractères spéciaux (garder espaces et ponctuation basique)
    text = re.sub(r"[^a-z0-9\s.!?\']", "", text)

    # Suppression espaces multiples
    text = re.sub(r"\s+", " ", text).strip()

    return text


# Appliquer le preprocessing
print("Preprocessing en cours...")
data["text_cleaned"] = data["text"].apply(preprocess_text)

# Suppression doublons
print(f"\nDoublons avant: {data.duplicated(subset=['text_cleaned']).sum()}")
data = data.drop_duplicates(subset=["text_cleaned"]).reset_index(drop=True)
print(f"Doublons après: {data.duplicated(subset=['text_cleaned']).sum()}")

# Suppression avis trop courts
print(f"\nAvis avant filtrage: {len(data)}")
data = data[data["text_cleaned"].str.len() > 5].reset_index(drop=True)
print(f"Avis après filtrage: {len(data)}")

# Afficher exemples
print("\nExemples après preprocessing:")
for i in range(3):
    print(f"Avant: {data['text'].iloc[i][:100]}...")
    print(f"Après: {data['text_cleaned'].iloc[i][:100]}...")
    print()

## Phase 5 : Feature Engineering

In [ ]:
# Split train/test
X_train, X_test, y_train, y_test = train_test_split(
    data["text_cleaned"],
    data["label"],
    test_size=0.2,
    random_state=42,
    stratify=data["label"],
)

print(f"Train set: {X_train.shape[0]}")
print(f"Test set: {X_test.shape[0]}")
print(f"Train class distribution: {y_train.value_counts()}")
print(f"Test class distribution: {y_test.value_counts()}")

# TF-IDF Vectorization
print("\n=== TF-IDF Vectorization ===")
tfidf = TfidfVectorizer(
    max_features=5000, min_df=2, max_df=0.8, ngram_range=(1, 2), lowercase=True
)

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

print(f"TF-IDF shape: {X_train_tfidf.shape}")
print(f"Nombre de features: {len(tfidf.get_feature_names_out())}")

# Top features
feature_names = np.array(tfidf.get_feature_names_out())
tfidf_array = X_train_tfidf.toarray()

print("\nTop features (par importance moyenne):")
feature_importance = tfidf_array.mean(axis=0)
top_indices = np.argsort(feature_importance)[-10:]
for idx in reversed(top_indices):
    print(f"{feature_names[idx]}: {feature_importance[idx]:.4f}")

## Phase 6 : Modélisation Comparative

In [ ]:
# 1. Logistic Regression
print("=== Logistic Regression ===")
lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train_tfidf, y_train)
y_pred_lr = lr.predict(X_test_tfidf)
y_proba_lr = lr.predict_proba(X_test_tfidf)[:, 1]

lr_acc = accuracy_score(y_test, y_pred_lr)
lr_f1 = f1_score(y_test, y_pred_lr)
lr_auc = roc_auc_score(y_test, y_proba_lr)
print(f"Accuracy: {lr_acc:.4f}")
print(f"F1 Score: {lr_f1:.4f}")
print(f"ROC-AUC: {lr_auc:.4f}")

# 2. Support Vector Machine
print("\n=== Support Vector Machine ===")
svm = SVC(kernel="linear", probability=True, random_state=42)
svm.fit(X_train_tfidf, y_train)
y_pred_svm = svm.predict(X_test_tfidf)
y_proba_svm = svm.predict_proba(X_test_tfidf)[:, 1]

svm_acc = accuracy_score(y_test, y_pred_svm)
svm_f1 = f1_score(y_test, y_pred_svm)
svm_auc = roc_auc_score(y_test, y_proba_svm)
print(f"Accuracy: {svm_acc:.4f}")
print(f"F1 Score: {svm_f1:.4f}")
print(f"ROC-AUC: {svm_auc:.4f}")

# 3. Random Forest
print("\n=== Random Forest ===")
rf = RandomForestClassifier(n_estimators=100, max_depth=20, random_state=42, n_jobs=-1)
rf.fit(X_train_tfidf, y_train)
y_pred_rf = rf.predict(X_test_tfidf)
y_proba_rf = rf.predict_proba(X_test_tfidf)[:, 1]

rf_acc = accuracy_score(y_test, y_pred_rf)
rf_f1 = f1_score(y_test, y_pred_rf)
rf_auc = roc_auc_score(y_test, y_proba_rf)
print(f"Accuracy: {rf_acc:.4f}")
print(f"F1 Score: {rf_f1:.4f}")
print(f"ROC-AUC: {rf_auc:.4f}")

# 4. Ensemble - Voting Classifier
print("\n=== Voting Classifier ===")
voting = VotingClassifier(
    estimators=[("lr", lr), ("svm", svm), ("rf", rf)], voting="soft"
)
voting.fit(X_train_tfidf, y_train)
y_pred_voting = voting.predict(X_test_tfidf)
y_proba_voting = voting.predict_proba(X_test_tfidf)[:, 1]

voting_acc = accuracy_score(y_test, y_pred_voting)
voting_f1 = f1_score(y_test, y_pred_voting)
voting_auc = roc_auc_score(y_test, y_proba_voting)
print(f"Accuracy: {voting_acc:.4f}")
print(f"F1 Score: {voting_f1:.4f}")
print(f"ROC-AUC: {voting_auc:.4f}")

## Phase 7 : Évaluation Complète

In [ ]:
# Résumé comparatif
results = pd.DataFrame(
    {
        "Model": ["Logistic Regression", "SVM", "Random Forest", "Voting Ensemble"],
        "Accuracy": [lr_acc, svm_acc, rf_acc, voting_acc],
        "F1 Score": [lr_f1, svm_f1, rf_f1, voting_f1],
        "ROC-AUC": [lr_auc, svm_auc, rf_auc, voting_auc],
    }
)

print("\n=== RÉSUMÉ COMPARATIF ===")
print(results.to_string(index=False))

# Sélectionner meilleur modèle
best_model_idx = results["F1 Score"].idxmax()
best_model_name = results.loc[best_model_idx, "Model"]
print(f"\nMeilleur modèle: {best_model_name}")

# Métriques détaillées pour meilleur modèle
if best_model_idx == 3:
    y_pred_best = y_pred_voting
    y_proba_best = y_proba_voting
elif best_model_idx == 2:
    y_pred_best = y_pred_rf
    y_proba_best = y_proba_rf
elif best_model_idx == 1:
    y_pred_best = y_pred_svm
    y_proba_best = y_proba_svm
else:
    y_pred_best = y_pred_lr
    y_proba_best = y_proba_lr

print("\n=== CLASSIFICATION REPORT ===")
print(classification_report(y_test, y_pred_best, target_names=["Négatif", "Positif"]))

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred_best)
print("\nMatrice de Confusion:")
print(cm)

## Phase 8 : Visualisations

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Comparaison modèles
results.set_index("Model")[["Accuracy", "F1 Score", "ROC-AUC"]].plot(
    kind="bar", ax=axes[0, 0]
)
axes[0, 0].set_title("Comparaison des Modèles")
axes[0, 0].set_ylabel("Score")
axes[0, 0].legend(loc="lower right")
axes[0, 0].set_xticklabels(axes[0, 0].get_xticklabels(), rotation=45, ha="right")

# 2. Confusion Matrix
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    ax=axes[0, 1],
    xticklabels=["Négatif", "Positif"],
    yticklabels=["Négatif", "Positif"],
)
axes[0, 1].set_title(f"Confusion Matrix - {best_model_name}")
axes[0, 1].set_ylabel("True Label")
axes[0, 1].set_xlabel("Predicted Label")

# 3. ROC Curves
fpr_lr, tpr_lr, _ = roc_curve(y_test, y_proba_lr)
fpr_svm, tpr_svm, _ = roc_curve(y_test, y_proba_svm)
fpr_rf, tpr_rf, _ = roc_curve(y_test, y_proba_rf)
fpr_voting, tpr_voting, _ = roc_curve(y_test, y_proba_voting)

axes[1, 0].plot(fpr_lr, tpr_lr, label=f"LR (AUC={lr_auc:.3f})")
axes[1, 0].plot(fpr_svm, tpr_svm, label=f"SVM (AUC={svm_auc:.3f})")
axes[1, 0].plot(fpr_rf, tpr_rf, label=f"RF (AUC={rf_auc:.3f})")
axes[1, 0].plot(
    fpr_voting, tpr_voting, label=f"Voting (AUC={voting_auc:.3f})", linewidth=2
)
axes[1, 0].plot([0, 1], [0, 1], "k--", label="Random")
axes[1, 0].set_xlabel("False Positive Rate")
axes[1, 0].set_ylabel("True Positive Rate")
axes[1, 0].set_title("ROC Curves")
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# 4. Prediction Distribution
axes[1, 1].hist(
    y_proba_best[y_test == 0], bins=30, alpha=0.6, label="Négatif", color="red"
)
axes[1, 1].hist(
    y_proba_best[y_test == 1], bins=30, alpha=0.6, label="Positif", color="green"
)
axes[1, 1].set_xlabel("Probabilité Prédite (Positif)")
axes[1, 1].set_ylabel("Fréquence")
axes[1, 1].set_title("Distribution des Probabilités")
axes[1, 1].legend()
axes[1, 1].axvline(x=0.5, color="black", linestyle="--", alpha=0.5)

plt.tight_layout()
plt.show()

## Phase 9 : Analyse des Erreurs

In [ ]:
# Identifier les erreurs
errors = y_test.values != y_pred_best
error_indices = np.where(errors)[0]

print(f"Nombre d'erreurs: {errors.sum()} / {len(y_test)}")
print(f"Taux d'erreur: {errors.mean():.2%}")

# Faux positifs et faux négatifs
false_positives = (y_test.values == 0) & (y_pred_best == 1)
false_negatives = (y_test.values == 1) & (y_pred_best == 0)

print(f"\nFaux positifs: {false_positives.sum()} (avis négatif prédit positif)")
print(f"Faux négatifs: {false_negatives.sum()} (avis positif prédit négatif)")

# Afficher exemples d'erreurs
print("\n=== Exemples de Faux Positifs ===")
fp_indices = np.where(false_positives)[0][:3]
for idx in fp_indices:
    print(f"Avis (réel: Négatif, prédit: Positif): {X_test.iloc[idx][:100]}...")
    print(f"Confiance: {y_proba_best[idx]:.3f}\n")

print("\n=== Exemples de Faux Négatifs ===")
fn_indices = np.where(false_negatives)[0][:3]
for idx in fn_indices:
    print(f"Avis (réel: Positif, prédit: Négatif): {X_test.iloc[idx][:100]}...")
    print(f"Confiance: {1 - y_proba_best[idx]:.3f}\n")

## Phase 10 : Sauvegarde du Modèle

In [ ]:
# Sauvegarder le modèle et le vectorizer
import json
from datetime import datetime

# Sauvegarder modèle
with open("best_sentiment_model.pkl", "wb") as f:
    pickle.dump(voting, f)

# Sauvegarder vectorizer
with open("tfidf_vectorizer.pkl", "wb") as f:
    pickle.dump(tfidf, f)

# Sauvegarder métadonnées
metadata = {
    "model_name": "Sentiment Analysis - Amazon Reviews",
    "best_model": "Voting Ensemble (LR + SVM + RF)",
    "accuracy": float(voting_acc),
    "f1_score": float(voting_f1),
    "roc_auc": float(voting_auc),
    "created_at": datetime.now().isoformat(),
    "features": "TF-IDF (5000 features, ngram=(1,2))",
    "training_samples": len(X_train),
    "test_samples": len(X_test),
}

with open("model_metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)

print("Modèle sauvegardé!")
print(json.dumps(metadata, indent=2))

## Phase 11 : Pipeline de Prédiction

In [ ]:
def predict_sentiment(
    text, model_path="best_sentiment_model.pkl", vectorizer_path="tfidf_vectorizer.pkl"
):
    """Prédire le sentiment d'un avis"""
    # Charger modèle et vectorizer
    with open(model_path, "rb") as f:
        model = pickle.load(f)
    with open(vectorizer_path, "rb") as f:
        vectorizer = pickle.load(f)

    # Preprocessing
    text_cleaned = preprocess_text(text)

    # Vectorisation
    text_vec = vectorizer.transform([text_cleaned])

    # Prédiction
    prediction = model.predict(text_vec)[0]
    probabilities = model.predict_proba(text_vec)[0]

    return {
        "text": text,
        "sentiment": "Positif" if prediction == 1 else "Négatif",
        "confidence": max(probabilities),
        "probability_negative": probabilities[0],
        "probability_positive": probabilities[1],
    }


# Tester le pipeline
test_texts = [
    "This product is absolutely amazing! I love it!",
    "Terrible quality, waste of money. Very disappointed.",
    "It's okay, nothing special but works as expected.",
]

print("\n=== TEST DU PIPELINE ===")
for text in test_texts:
    result = predict_sentiment(text)
    print(f"\nTexte: {result['text'][:60]}...")
    print(f"Sentiment: {result['sentiment']}")
    print(f"Confiance: {result['confidence']:.3f}")
    print(f"Prob Négatif: {result['probability_negative']:.3f}")
    print(f"Prob Positif: {result['probability_positive']:.3f}")